# 02 - Activation Patching vs Path Patching

**Question:** Is a component causal, and through which downstream receiver is its effect mediated?

**Prediction:** Patching `source` restores the behavior. The `source -> causal` path carries the effect; `source -> nuisance` does not.

**Expected compute:** CPU, seconds.

**Maturity:** module activation patching is Stable; module path patching is Research.


In [ ]:
from neuros_mechint.benchmarks import GroundTruthCausalMLP, make_ground_truth_pair
from neuros_mechint.circuits import ModuleActivationPatcher, PathPatcher

model = GroundTruthCausalMLP().eval()
pair = make_ground_truth_pair()
metric = lambda output: output.mean()


In [ ]:
activation = ModuleActivationPatcher(
    model, metric, layers_to_patch=["source", "causal", "nuisance"]
).patch_all(pair.clean, pair.corrupted)

for effect in activation.effects:
    print(effect.layer_name, effect.direct_effect)


In [ ]:
paths = PathPatcher(
    model, metric, layers_to_patch=["source", "causal", "nuisance"]
).patch_all_paths(
    pair.clean, pair.corrupted,
    senders=["source"], receivers=["causal", "nuisance"],
)

for effect in paths.effects:
    print(f"{effect.sender} -> {effect.receiver}: {effect.mediated_effect}")


## Why the distinction matters

Single-component activation patching tells us that a module contains causally useful state. It does not prove which downstream path carries that state.

The path experiment patches the sender, records the receiver state produced by that intervention, then transplants only that receiver state into the original corrupted run. This asks a narrower receiver-mediated question.

### Next exercise

Add a second nonzero downstream branch and test whether the two receiver-mediated effects reconstruct the sender's total effect. Then create a nonlinear interaction where they do not add. What does that teach you about circuit decomposition?
